# Image Analyser — Training on Colab GPU

**Before running:**
1. Upload `training_dataset_v4_train_test.zip` to your Google Drive
2. Make sure your latest code is pushed to GitHub
3. Set Runtime → Change runtime type → **T4 GPU**

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/ai-property-triage'
BRANCH   = 'feature/image-analyser'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/Muhammadegb1/ai-property-triage.git {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

In [ ]:
!pip install -q torch torchvision pillow numpy

In [ ]:
# Extract images from Drive — labels.csv is already committed in the repo
import zipfile

ZIP_PATH = '/content/drive/MyDrive/training_dataset_v4_train_test.zip'
RAW_DIR  = '/content/ai-property-triage/services/image_analyser/data/raw'

os.makedirs(RAW_DIR, exist_ok=True)
print('Extracting images...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(RAW_DIR)

total = sum(len(files) for _, _, files in os.walk(RAW_DIR))
print(f'Extracted {total} files')

In [ ]:
%cd /content/ai-property-triage/services/image_analyser
!python train.py

In [ ]:
import shutil

CHECKPOINT_SRC = '/content/ai-property-triage/services/image_analyser/checkpoints/best_model.pth'
REPORT_SRC     = '/content/ai-property-triage/services/image_analyser/checkpoints/training_report.txt'
DRIVE_DEST     = '/content/drive/MyDrive/ai_property_triage_checkpoints/'

os.makedirs(DRIVE_DEST, exist_ok=True)

if os.path.exists(CHECKPOINT_SRC):
    shutil.copy(CHECKPOINT_SRC, DRIVE_DEST)
    print(f'Saved best_model.pth to {DRIVE_DEST}')
else:
    print('ERROR: best_model.pth not found')

if os.path.exists(REPORT_SRC):
    shutil.copy(REPORT_SRC, DRIVE_DEST)
    print('\n--- Training Report ---')
    with open(REPORT_SRC) as f:
        print(f.read())